<a href="https://colab.research.google.com/github/ancestor9/2026_Fall_Learning-Langchain-AI-Agent/blob/main/excercise/gemini_pydantic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pydantic to structure output from LLMs
- Pydantic을 사용하여 Gemini (LLM) 모델의 출력을 구조화하고, 이 구조화된 데이터를 파이썬에서 효과적으로 다루는 방법을 설명
- Gemini 모델에게 JSON 형식의 데이터를 생성하도록 요청하고, 이 JSON 문자열을 Pydantic 모델을 사용하여 강력하고 다루기 쉬운 Python 객체로 **역직렬화(Deserialize)** 하는 과정
- LLM이 생성하는 비정형(unstructured) 출력을 Pydantic의 강력한 타입 시스템을 통해 정형(structured) 데이터로 변환하여, 후속 작업(분석, 저장, 처리)을 안전하고 쉽게 수행할 수 있도록 해주는 모범적인 예시

In [2]:
from google import genai
import os

from google.colab import userdata
GEMINI_API_KEY = userdata.get('Gemini-API-Key')

client = genai.Client(api_key= GEMINI_API_KEY)

response = client.models.generate_content(
    model="gemini-3.5-flash-lite", contents="Explain how AI works in a few words"
)
print(response.text)

AI finds patterns in massive amounts of data to make predictions or decisions.


In [3]:
response = client.models.generate_content(
    model="gemini-3.5-flash-lite",
    contents="""You are a helpful assistant. I need you to create a JSON object representing a library.
    The library's name should be 'Coolu Libraru' and have the fields name and books that contains a list of book.
    Each book should have a 'title', 'author', and 'year' field. Make sure the output is a single, valid JSON object. Give me 10 books.
    Remove ```json and ``` """,
)

response.text

'{\n  "name": "Coolu Libraru",\n  "books": [\n    {\n      "title": "The Shadow of the Wind",\n      "author": "Carlos Ruiz Zafón",\n      "year": 2001\n    },\n    {\n      "title": "The Hitchhiker\'s Guide to the Galaxy",\n      "author": "Douglas Adams",\n      "year": 1979\n    },\n    {\n      "title": "Dune",\n      "author": "Frank Herbert",\n      "year": 1965\n    },\n    {\n      "title": "Neuromancer",\n      "author": "William Gibson",\n      "year": 1984\n    },\n    {\n      "title": "The Name of the Wind",\n      "author": "Patrick Rothfuss",\n      "year": 2007\n    },\n    {\n      "title": "The Ocean at the End of the Lane",\n      "author": "Neil Gaiman",\n      "year": 2013\n    },\n    {\n      "title": "Snow Crash",\n      "author": "Neal Stephenson",\n      "year": 1992\n    },\n    {\n      "title": "Project Hail Mary",\n      "author": "Andy Weir",\n      "year": 2021\n    },\n    {\n      "title": "Brave New World",\n      "author": "Aldous Huxley",\n      "ye

In [4]:
print(response.text)

{
  "name": "Coolu Libraru",
  "books": [
    {
      "title": "The Shadow of the Wind",
      "author": "Carlos Ruiz Zafón",
      "year": 2001
    },
    {
      "title": "The Hitchhiker's Guide to the Galaxy",
      "author": "Douglas Adams",
      "year": 1979
    },
    {
      "title": "Dune",
      "author": "Frank Herbert",
      "year": 1965
    },
    {
      "title": "Neuromancer",
      "author": "William Gibson",
      "year": 1984
    },
    {
      "title": "The Name of the Wind",
      "author": "Patrick Rothfuss",
      "year": 2007
    },
    {
      "title": "The Ocean at the End of the Lane",
      "author": "Neil Gaiman",
      "year": 2013
    },
    {
      "title": "Snow Crash",
      "author": "Neal Stephenson",
      "year": 1992
    },
    {
      "title": "Project Hail Mary",
      "author": "Andy Weir",
      "year": 2021
    },
    {
      "title": "Brave New World",
      "author": "Aldous Huxley",
      "year": 1932
    },
    {
      "title": "The Marti

- model_validate_json(): Gemini가 반환한 JSON 문자열을 곧바로 Library 모델 객체로 변환하고 유효성 검사를 수행
- JSON 문자열은 객체 지향 방식으로 다룰 수 있는 library 객체로 변경

In [5]:
from pydantic import BaseModel, Field
from typing import List
from datetime import datetime

class Book(BaseModel):
    title: str
    author: str
    year: int = Field(gt = 1000, lt = datetime.now().year)

class Library(BaseModel):
    name: str
    books: List[Book]

# JSON 문자열을 파싱하는 동시에 Python 객체로 유효성 검사(Validation) 및 타입 변환을 자동으로 수행
library = Library.model_validate_json(response.text)
library

Library(name='Coolu Libraru', books=[Book(title='The Shadow of the Wind', author='Carlos Ruiz Zafón', year=2001), Book(title="The Hitchhiker's Guide to the Galaxy", author='Douglas Adams', year=1979), Book(title='Dune', author='Frank Herbert', year=1965), Book(title='Neuromancer', author='William Gibson', year=1984), Book(title='The Name of the Wind', author='Patrick Rothfuss', year=2007), Book(title='The Ocean at the End of the Lane', author='Neil Gaiman', year=2013), Book(title='Snow Crash', author='Neal Stephenson', year=1992), Book(title='Project Hail Mary', author='Andy Weir', year=2021), Book(title='Brave New World', author='Aldous Huxley', year=1932), Book(title='The Martian Chronicles', author='Ray Bradbury', year=1950)])

In [12]:
# 객체(Object)가 가지고 있는 속성(Attributes)과 그 값을 딕셔너리(dictionary) 형태로 보여주는 특별한 속성
library.__dict__

{'name': 'Coolu Libraru',
 'books': [Book(title='The Shadow of the Wind', author='Carlos Ruiz Zafón', year=2001),
  Book(title="The Hitchhiker's Guide to the Galaxy", author='Douglas Adams', year=1979),
  Book(title='Dune', author='Frank Herbert', year=1965),
  Book(title='Neuromancer', author='William Gibson', year=1984),
  Book(title='The Name of the Wind', author='Patrick Rothfuss', year=2007),
  Book(title='The Ocean at the End of the Lane', author='Neil Gaiman', year=2013),
  Book(title='Snow Crash', author='Neal Stephenson', year=1992),
  Book(title='Project Hail Mary', author='Andy Weir', year=2021),
  Book(title='Brave New World', author='Aldous Huxley', year=1932),
  Book(title='The Martian Chronicles', author='Ray Bradbury', year=1950)]}

In [13]:
type(library)

__main__.Library

In [14]:
# isinstance(library, BaseModel)은 library라는 객체가 BaseModel 클래스(또는 그 하위 클래스)의 인스턴스가 맞는지 확인
isinstance(library, BaseModel)

True

In [9]:
# 도서관 이름 접근
library.name

'Coolu Libraru'

In [15]:
# # 책 리스트 접근
library.books

[Book(title='The Shadow of the Wind', author='Carlos Ruiz Zafón', year=2001),
 Book(title="The Hitchhiker's Guide to the Galaxy", author='Douglas Adams', year=1979),
 Book(title='Dune', author='Frank Herbert', year=1965),
 Book(title='Neuromancer', author='William Gibson', year=1984),
 Book(title='The Name of the Wind', author='Patrick Rothfuss', year=2007),
 Book(title='The Ocean at the End of the Lane', author='Neil Gaiman', year=2013),
 Book(title='Snow Crash', author='Neal Stephenson', year=1992),
 Book(title='Project Hail Mary', author='Andy Weir', year=2021),
 Book(title='Brave New World', author='Aldous Huxley', year=1932),
 Book(title='The Martian Chronicles', author='Ray Bradbury', year=1950)]

extract titles into a list

In [16]:
# 리스트 컴프리헨션을 사용한 데이터 추출 및 필터링
titles = [book.title for book in library.books]
titles

['The Shadow of the Wind',
 "The Hitchhiker's Guide to the Galaxy",
 'Dune',
 'Neuromancer',
 'The Name of the Wind',
 'The Ocean at the End of the Lane',
 'Snow Crash',
 'Project Hail Mary',
 'Brave New World',
 'The Martian Chronicles']

extract titles with books after certain year

In [17]:
newer_books = [(book.title, book.year) for book in library.books if book.year > 1950]
newer_books

[('The Shadow of the Wind', 2001),
 ("The Hitchhiker's Guide to the Galaxy", 1979),
 ('Dune', 1965),
 ('Neuromancer', 1984),
 ('The Name of the Wind', 2007),
 ('The Ocean at the End of the Lane', 2013),
 ('Snow Crash', 1992),
 ('Project Hail Mary', 2021)]

to get back json data use model_dump()
- **직렬화**: library.model_dump()를 사용하여 Pydantic 객체를 다시 Python 딕셔너리로 변환하거나, library.model_dump_json()를 사용하여 JSON 파일로 저장

In [18]:
library.model_dump()

{'name': 'Coolu Libraru',
 'books': [{'title': 'The Shadow of the Wind',
   'author': 'Carlos Ruiz Zafón',
   'year': 2001},
  {'title': "The Hitchhiker's Guide to the Galaxy",
   'author': 'Douglas Adams',
   'year': 1979},
  {'title': 'Dune', 'author': 'Frank Herbert', 'year': 1965},
  {'title': 'Neuromancer', 'author': 'William Gibson', 'year': 1984},
  {'title': 'The Name of the Wind',
   'author': 'Patrick Rothfuss',
   'year': 2007},
  {'title': 'The Ocean at the End of the Lane',
   'author': 'Neil Gaiman',
   'year': 2013},
  {'title': 'Snow Crash', 'author': 'Neal Stephenson', 'year': 1992},
  {'title': 'Project Hail Mary', 'author': 'Andy Weir', 'year': 2021},
  {'title': 'Brave New World', 'author': 'Aldous Huxley', 'year': 1932},
  {'title': 'The Martian Chronicles', 'author': 'Ray Bradbury', 'year': 1950}]}

In [19]:
library.model_dump_json()

'{"name":"Coolu Libraru","books":[{"title":"The Shadow of the Wind","author":"Carlos Ruiz Zafón","year":2001},{"title":"The Hitchhiker\'s Guide to the Galaxy","author":"Douglas Adams","year":1979},{"title":"Dune","author":"Frank Herbert","year":1965},{"title":"Neuromancer","author":"William Gibson","year":1984},{"title":"The Name of the Wind","author":"Patrick Rothfuss","year":2007},{"title":"The Ocean at the End of the Lane","author":"Neil Gaiman","year":2013},{"title":"Snow Crash","author":"Neal Stephenson","year":1992},{"title":"Project Hail Mary","author":"Andy Weir","year":2021},{"title":"Brave New World","author":"Aldous Huxley","year":1932},{"title":"The Martian Chronicles","author":"Ray Bradbury","year":1950}]}'

output json file

In [20]:
with open("library.json", "w") as json_file:
    json_file.write(library.model_dump_json())

create pandas dataframe
- 데이터프레임 생성: 추출된 데이터를 사용하여 Pandas DataFrame을 생성, 이는 구조화된 데이터를 데이터 분석에 활용하는 일반적인 방법

In [21]:
import pandas as pd

titles = [book.title for book in library.books]
years = [book.year for book in library.books]
authors = [book.author for book in library.books]

pd.DataFrame({"title": titles, "year": years, "author": authors})

,title,year,author
0,The Shadow of the Wind,2001,Carlos Ruiz Zafón
1,The Hitchhiker's Guide to the Galaxy,1979,Douglas Adams
2,Dune,1965,Frank Herbert
3,Neuromancer,1984,William Gibson
4,The Name of the Wind,2007,Patrick Rothfuss
5,The Ocean at the End of the Lane,2013,Neil Gaiman
6,Snow Crash,1992,Neal Stephenson
7,Project Hail Mary,2021,Andy Weir
8,Brave New World,1932,Aldous Huxley
9,The Martian Chronicles,1950,Ray Bradbury
